# day.csv: Data Boss session

Exported from Data Boss. Run the cells from top to bottom next to the data file to repeat every cleaning step and chart. Each step has the message the app showed when it was applied.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Load the data

Reads `day.csv` exactly as Data Boss did.

In [ ]:
df = pd.read_csv('day.csv')
df.shape

## Cleaning steps

### Step 1

Converted 1 column to dates.

Rows × columns: 731 × 16 → 731 × 16

In [ ]:
for col in ['dteday']:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=False, format='mixed')
df.shape

### Step 2

Added column 'weather' = CASE weathersit WHEN 1 THEN 'clear' WHEN 2 THEN 'mist / cloudy' WHEN 3 THEN 'light rain / snow' ELSE 'heavy rain / snow' END.

Rows × columns: 731 × 16 → 731 × 17

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['weather'] = con.sql("""SELECT (CASE weathersit WHEN 1 THEN 'clear' WHEN 2 THEN 'mist / cloudy' WHEN 3 THEN 'light rain / snow' ELSE 'heavy rain / snow' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 3

Added column 'day_type' = CASE WHEN workingday = 1 THEN 'working day' ELSE 'weekend / holiday' END.

Rows × columns: 731 × 17 → 731 × 18

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['day_type'] = con.sql("""SELECT (CASE WHEN workingday = 1 THEN 'working day' ELSE 'weekend / holiday' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 4

Added column 'year' = CASE yr WHEN 0 THEN '2011' ELSE '2012' END.

Rows × columns: 731 × 18 → 731 × 19

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['year'] = con.sql("""SELECT (CASE yr WHEN 0 THEN '2011' ELSE '2012' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 5

Added column 'temp_c' = ROUND(temp * 47 - 8, 1).

Rows × columns: 731 × 19 → 731 × 20

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['temp_c'] = con.sql("""SELECT (ROUND(temp * 47 - 8, 1)) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 6

Added 1 column from dteday: dteday_weekday.

Rows × columns: 731 × 20 → 731 × 21

In [ ]:
df['dteday_weekday'] = df['dteday'].dt.day_name().astype(pd.CategoricalDtype(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], ordered=True))
df.shape

### Step 7

Dropped 9 column(s): instant, season, yr, mnth, weathersit, temp, atemp, hum, windspeed.

Rows × columns: 731 × 21 → 731 × 12

In [ ]:
df = df.drop(columns=['instant', 'season', 'yr', 'mnth', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed'])
df.shape

## Result

In [ ]:
df = df.reset_index(drop=True)
df.head(10)

## Charts

Drawn from the cleaned data above.

### Average casual riders by weekday

- Highest: Saturday (1,465). Lowest: Wednesday (551).
- The highest group is 2.7× the lowest.

In [ ]:
s = df.groupby('dteday_weekday')['casual'].mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9.0, 5.0))
ax.bar(s.index.astype(str), s.values)
plt.tight_layout()
plt.show()

### Daily rentals by weather

- Highest median: clear (4,844). Lowest: light rain / snow (1,817).
- Overall median 4,548; middle 50% between 3,152 and 5,956.
- No points outside the whiskers.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.0))
order = df.groupby('weather')['cnt'].median().sort_values(ascending=False).index[:12]
sns.boxplot(data=df[df['weather'].isin(order)], x='weather', y='cnt', order=order, ax=ax)
plt.tight_layout()
plt.show()

### Daily rentals against temperature (°C)

- Pearson r = 0.63: strong positive relationship (731 points).
- Trend line: each +1 in temp_c goes with about +141 in cnt.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.0))
sns.scatterplot(data=df, x='temp_c', y='cnt', hue='year', alpha=0.55, s=20, ax=ax)
d = df[['temp_c', 'cnt']].dropna()
m, b = np.polyfit(d['temp_c'], d['cnt'], 1)
xs = np.linspace(d['temp_c'].min(), d['temp_c'].max(), 100)
ax.plot(xs, m * xs + b, linestyle='--')
plt.tight_layout()
plt.show()

## Save the cleaned data

In [ ]:
df.to_csv('day_cleaned.csv', index=False)